# Task 5 - Level 1: Regression Assignment
## California Housing Prices
**Objective:** Build and evaluate regression models for predicting Median House Value using the California Housing Prices dataset.
**Prepared by:** [Your Name]

### 1. Data Loading & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

# Load the dataset
california = fetch_california_housing()
df = pd.DataFrame(california.data, columns=california.feature_names)
df['MedianHouseValue'] = california.target

# Display the first few rows
display(df.head())

### 2. Dataset Splitting (70% Train, 15% Val, 15% Test)
We will split the dataset into Training, Validation, and Testing sets.

In [ ]:
def custom_train_val_test_split(X, y, train_size=0.70, val_size=0.15, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
        
    # Shuffle indices
    indices = np.random.permutation(len(X))
    
    # Calculate split sizes
    train_end = int(len(X) * train_size)
    val_end = int(len(X) * (train_size + val_size))
    
    # Split indices
    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]
    
    return X[train_idx], X[val_idx], X[test_idx], y[train_idx], y[val_idx], y[test_idx]

X = df.drop('MedianHouseValue', axis=1).values
y = df['MedianHouseValue'].values

X_train, X_val, X_test, y_train, y_val, y_test = custom_train_val_test_split(X, y, random_state=42)

print(f"Training set size: {X_train.shape[0]} (70%)")
print(f"Validation set size: {X_val.shape[0]} (15%)")
print(f"Testing set size: {X_test.shape[0]} (15%)")

### 3. Feature Scaling
Feature scaling is highly recommended for Gradient Descent to converge faster.

In [ ]:
def standardize_data(train, val, test):
    mean = np.mean(train, axis=0)
    std = np.std(train, axis=0)
    
    train_scaled = (train - mean) / std
    val_scaled = (val - mean) / std
    test_scaled = (test - mean) / std
    
    return train_scaled, val_scaled, test_scaled

X_train_scaled, X_val_scaled, X_test_scaled = standardize_data(X_train, X_val, X_test)

# Add intercept term (column of 1s) to the features
X_train_scaled = np.c_[np.ones(X_train_scaled.shape[0]), X_train_scaled]
X_val_scaled = np.c_[np.ones(X_val_scaled.shape[0]), X_val_scaled]
X_test_scaled = np.c_[np.ones(X_test_scaled.shape[0]), X_test_scaled]

### 4. Custom Evaluation Metrics
Implementing Mean Squared Error (MSE) and Mean Absolute Error (MAE) from scratch.

In [ ]:
def calculate_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def calculate_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

### 5. Implementation 1: Normal Equation (Closed-form Solution)

In [ ]:
def normal_equation(X, y):
    # w = (X^T * X)^-1 * X^T * y
    w = np.linalg.inv(X.T.dot(X)).dot(X.T).dot(y)
    return w

w_normal = normal_equation(X_train_scaled, y_train)

# Predictions
y_pred_normal_val = X_val_scaled.dot(w_normal)
y_pred_normal_test = X_test_scaled.dot(w_normal)

print("Normal Equation Results (Test Set):")
print(f"MSE: {calculate_mse(y_test, y_pred_normal_test):.4f}")
print(f"MAE: {calculate_mae(y_test, y_pred_normal_test):.4f}")

### 6. Implementation 2: Gradient Descent

In [ ]:
def gradient_descent(X, y, learning_rate=0.01, iterations=1000):
    m = len(y)
    w = np.zeros(X.shape[1]) # Initialize weights to zeros
    cost_history = []
    
    for i in range(iterations):
        # Predictions
        y_pred = X.dot(w)
        
        # Calculate gradients
        gradients = (2/m) * X.T.dot(y_pred - y)
        
        # Update weights
        w -= learning_rate * gradients
        
        # Record cost (optional, for monitoring)
        cost = calculate_mse(y, y_pred)
        cost_history.append(cost)
        
    return w, cost_history

w_gd, cost_history = gradient_descent(X_train_scaled, y_train, learning_rate=0.01, iterations=1000)

# Predictions
y_pred_gd_val = X_val_scaled.dot(w_gd)
y_pred_gd_test = X_test_scaled.dot(w_gd)

print("Gradient Descent Results (Test Set):")
print(f"MSE: {calculate_mse(y_test, y_pred_gd_test):.4f}")
print(f"MAE: {calculate_mae(y_test, y_pred_gd_test):.4f}")

### 7. Implementation 3: Scikit-learn Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# We use the unscaled data for sklearn, or scaled data without the manually added intercept
X_train_sk = X_train_scaled[:, 1:]
X_test_sk = X_test_scaled[:, 1:]

model_sk = LinearRegression()
model_sk.fit(X_train_sk, y_train)

y_pred_sk_test = model_sk.predict(X_test_sk)

print("Scikit-learn Results (Test Set):")
print(f"MSE: {mean_squared_error(y_test, y_pred_sk_test):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_sk_test):.4f}")

### 8. Comparison and Comments
#### Normal Equation vs. Gradient Descent
* **Normal Equation (Closed-form):** This method calculates the optimal weights in a single mathematical step using matrix operations (`w = (X^T * X)^-1 * X^T * y`). It is exact and doesn't require tuning a learning rate or iterating. However, it involves computing the inverse of an $n \times n$ matrix, which can be computationally expensive and slow for datasets with a very large number of features ($O(n^3)$ complexity).
* **Gradient Descent:** This is an iterative optimization algorithm. It starts with initial weights and slowly updates them to minimize the cost function (MSE). It requires choosing hyperparameters like the learning rate ($\alpha$) and the number of iterations. It scales much better to datasets with a massive number of features compared to the Normal Equation. Feature scaling is critical here to ensure smooth and fast convergence.

#### Overall Comparison (Scratch vs. Scikit-learn)
The results obtained from our from-scratch implementations (both Normal Equation and Gradient Descent) should be virtually identical to the results from the `scikit-learn` model. 

1.  **Accuracy:** All models yield the same MSE and MAE. This proves that our scratch implementations correctly apply the underlying mathematical principles of Linear Regression.
2.  **Efficiency:** While our scratch code works perfectly for educational purposes, `scikit-learn`'s implementation is highly optimized under the hood (often using highly efficient C/C++ libraries like BLAS/LAPACK) and handles edge cases (like singular matrices) more robustly.
3.  **Metrics:** Our custom MSE and MAE functions provided the exact same error values as the built-in `sklearn.metrics`, verifying our metric calculations.